In [1]:
!pip install imageio

In [2]:
!pip install scikit-learn matplotlib numpy

In [3]:
# Cell 1: path + global imports
import sys
sys.path.append(r"C:\Preet\CV Project\VAE_AC\100 epochs")

import os, gc, time
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from tqdm import tqdm
import imageio

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torch.nn.functional import softplus

from torch.optim import Adam
from torch.amp import GradScaler, autocast

from sklearn.decomposition import PCA

print("Python:", sys.executable)
print("Torch:", torch.__version__, "CUDA:", torch.cuda.is_available())
gc.collect()


Python: c:\Preet\CV Project\VAE_AC\vaeac_env_preet\Scripts\python.exe
Torch: 2.7.0+cu128 CUDA: True


40

In [4]:
# Cell 2: import your mask generator module (file must exist at project root)
# Make sure you have C:\Preet\CV Project\VAE_AC\mask_generators.py saved beforehand.
from mask_generators import ImageMaskGenerator
print("mask_generators.ImageMaskGenerator imported.")


mask_generators.ImageMaskGenerator imported.


In [5]:
# Cell 3: settings + helper functions
PROJECT_ROOT = r"C:\Preet\CV Project\VAE_AC\100 epochs"
IMG_FOLDER = os.path.join(PROJECT_ROOT, "img_align_celeba")
RESULTS_DIR = os.path.join(PROJECT_ROOT, "results")
LATENT_DIR = os.path.join(PROJECT_ROOT, "latent_history")
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(LATENT_DIR, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

BATCH_SIZE = 16
EPOCHS = 50   # change if you want fewer
SAVE_FIXED_COUNT = 16  # how many fixed samples to save

def tensor_to_pil(t):
    t = t.detach().cpu()
    # assume normalized in [-1,1]
    if t.min() < -0.1:
        t = t * 0.5 + 0.5
    t = t.clamp(0,1)
    arr = (t.permute(1,2,0).numpy()*255).astype(np.uint8)
    return Image.fromarray(arr)

def save_image_tensor(path, tensor):
    pil = tensor_to_pil(tensor)
    pil.save(path)


Using device: cuda


In [6]:
# Cell 4: dataset and transform
transform = transforms.Compose([
    transforms.Resize((128,128)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))
])

class CelebA(Dataset):
    def __init__(self, folder):
        self.folder = folder
        self.files = sorted(os.listdir(folder))
    def __len__(self):
        return len(self.files)
    def __getitem__(self, idx):
        img = Image.open(os.path.join(self.folder, self.files[idx])).convert("RGB")
        return transform(img)

train_ds = CelebA(IMG_FOLDER)
# Use num_workers=0 in notebook on Windows
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=False)
print("Dataset size:", len(train_ds))


Dataset size: 30096


In [7]:
# Cell 5: NN utility classes (ResBlock, SkipConnection, MemoryLayer)
class ResBlock(nn.Module):
    def __init__(self, outer_dim, inner_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.BatchNorm2d(outer_dim),
            nn.LeakyReLU(),
            nn.Conv2d(outer_dim, inner_dim, 1),
            nn.BatchNorm2d(inner_dim),
            nn.LeakyReLU(),
            nn.Conv2d(inner_dim, inner_dim, 3, 1, 1),
            nn.BatchNorm2d(inner_dim),
            nn.LeakyReLU(),
            nn.Conv2d(inner_dim, outer_dim, 1),
        )
    def forward(self, x):
        return x + self.net(x)

class SkipConnection(nn.Module):
    def __init__(self, *args):
        super().__init__()
        self.inner = nn.Sequential(*args)
    def forward(self, x):
        return x + self.inner(x)

class MemoryLayer(nn.Module):
    storage = {}
    def __init__(self, id, output=False, add=False):
        super().__init__()
        self.id = id; self.output = output; self.add = add
    def forward(self, x):
        if not self.output:
            MemoryLayer.storage[self.id] = x
            return x
        else:
            stored = MemoryLayer.storage[self.id]
            if not self.add:
                return torch.cat([x, stored], 1)
            else:
                return x + stored


In [8]:
# Cell 6: robust normal parse, sampling and KL calculation helpers
import math

def parse_params_to_mu_sigma(params, min_sigma=1e-2):
    """
    params: (B, 2*C, H, W)
    Returns:
      mu:    (B,C,H,W)
      sigma: (B,C,H,W)   with safe lower bound
    """
    B, C2, H, W = params.shape
    assert C2 % 2 == 0, "channels in params must be even"
    C = C2 // 2

    mu = params[:, :C, :, :].contiguous()
    sigma_params = params[:, C:, :, :].contiguous()

    # ---- Stability fix: avoid sigma collapse ----
    sigma = softplus(sigma_params) + 1e-2       # ensures sigma >= 0.01

    # Extra clamp to avoid INF
    sigma = sigma.clamp(min=min_sigma, max=50.0)

    return mu, sigma

def sample_from_mu_sigma(mu, sigma):
    eps = torch.randn_like(mu)
    return mu + eps * sigma


def kl_normal(mu_q, sigma_q, mu_p, sigma_p):
    # KL between two diagonal Gaussians
    sq = sigma_q
    sp = sigma_p

    term = (
        torch.log(sp) - torch.log(sq)
        + (sq**2 + (mu_q - mu_p)**2) / (2 * sp**2)
        - 0.5
    )

    return term.view(term.size(0), -1).sum(-1)



In [9]:
# Cell 7: build proposal_network, prior_network, generative_network (as in your repo)
def MLPBlock(dim):
    return SkipConnection(
        nn.BatchNorm2d(dim),
        nn.LeakyReLU(),
        nn.Conv2d(dim, dim, 1)
    )

proposal_network = nn.Sequential(
    nn.Conv2d(6, 8, 1),
    ResBlock(8, 8), ResBlock(8, 8), ResBlock(8, 8), ResBlock(8, 8),
    nn.AvgPool2d(2, 2),
    ResBlock(8, 8), ResBlock(8, 8), ResBlock(8, 8), ResBlock(8, 8),
    nn.AvgPool2d(2, 2), nn.Conv2d(8, 16, 1),
    ResBlock(16, 8), ResBlock(16, 8), ResBlock(16, 8), ResBlock(16, 8),
    nn.AvgPool2d(2, 2), nn.Conv2d(16, 32, 1),
    ResBlock(32, 16), ResBlock(32, 16), ResBlock(32, 16), ResBlock(32, 16),
    nn.AvgPool2d(2, 2), nn.Conv2d(32, 64, 1),
    ResBlock(64, 32), ResBlock(64, 32), ResBlock(64, 32), ResBlock(64, 32),
    nn.AvgPool2d(2, 2), nn.Conv2d(64, 128, 1),
    ResBlock(128, 64), ResBlock(128, 64),
    ResBlock(128, 64), ResBlock(128, 64),
    nn.AvgPool2d(2, 2), nn.Conv2d(128, 256, 1),
    ResBlock(256, 128), ResBlock(256, 128),
    ResBlock(256, 128), ResBlock(256, 128),
    nn.AvgPool2d(2, 2), nn.Conv2d(256, 512, 1),
    MLPBlock(512), MLPBlock(512), MLPBlock(512), MLPBlock(512),
)

prior_network = nn.Sequential(
    MemoryLayer('#0'),
    nn.Conv2d(6, 8, 1),
    ResBlock(8, 8), ResBlock(8, 8), ResBlock(8, 8), ResBlock(8, 8),
    MemoryLayer('#1'),
    nn.AvgPool2d(2, 2),
    ResBlock(8, 8), ResBlock(8, 8), ResBlock(8, 8), ResBlock(8, 8),
    MemoryLayer('#2'),
    nn.AvgPool2d(2, 2), nn.Conv2d(8, 16, 1),
    ResBlock(16, 8), ResBlock(16, 8), ResBlock(16, 8), ResBlock(16, 8),
    MemoryLayer('#3'),
    nn.AvgPool2d(2, 2), nn.Conv2d(16, 32, 1),
    ResBlock(32, 16), ResBlock(32, 16), ResBlock(32, 16), ResBlock(32, 16),
    MemoryLayer('#4'),
    nn.AvgPool2d(2, 2), nn.Conv2d(32, 64, 1),
    ResBlock(64, 32), ResBlock(64, 32), ResBlock(64, 32), ResBlock(64, 32),
    MemoryLayer('#5'),
    nn.AvgPool2d(2, 2), nn.Conv2d(64, 128, 1),
    ResBlock(128, 64), ResBlock(128, 64),
    ResBlock(128, 64), ResBlock(128, 64),
    MemoryLayer('#6'),
    nn.AvgPool2d(2, 2), nn.Conv2d(128, 256, 1),
    ResBlock(256, 128), ResBlock(256, 128),
    ResBlock(256, 128), ResBlock(256, 128),
    MemoryLayer('#7'),
    nn.AvgPool2d(2, 2), nn.Conv2d(256, 512, 1),
    MLPBlock(512), MLPBlock(512), MLPBlock(512), MLPBlock(512),
)

generative_network = nn.Sequential(
    nn.Conv2d(256, 256, 1),
    MLPBlock(256), MLPBlock(256), MLPBlock(256), MLPBlock(256),
    nn.Conv2d(256, 128, 1), nn.Upsample(scale_factor=2),
    MemoryLayer('#7', output=True), nn.Conv2d(384, 128, 1),
    ResBlock(128, 64), ResBlock(128, 64),
    ResBlock(128, 64), ResBlock(128, 64),
    nn.Conv2d(128, 64, 1), nn.Upsample(scale_factor=2),
    MemoryLayer('#6', output=True), nn.Conv2d(192, 64, 1),
    ResBlock(64, 32), ResBlock(64, 32), ResBlock(64, 32), ResBlock(64, 32),
    nn.Conv2d(64, 32, 1), nn.Upsample(scale_factor=2),
    MemoryLayer('#5', output=True), nn.Conv2d(96, 32, 1),
    ResBlock(32, 16), ResBlock(32, 16), ResBlock(32, 16), ResBlock(32, 16),
    nn.Conv2d(32, 16, 1), nn.Upsample(scale_factor=2),
    MemoryLayer('#4', output=True), nn.Conv2d(48, 16, 1),
    ResBlock(16, 8), ResBlock(16, 8), ResBlock(16, 8), ResBlock(16, 8),
    nn.Conv2d(16, 8, 1), nn.Upsample(scale_factor=2),
    MemoryLayer('#3', output=True), nn.Conv2d(24, 8, 1),
    ResBlock(8, 8), ResBlock(8, 8), ResBlock(8, 8), ResBlock(8, 8),
    nn.Upsample(scale_factor=2),
    MemoryLayer('#2', output=True), nn.Conv2d(16, 8, 1),
    ResBlock(8, 8), ResBlock(8, 8), ResBlock(8, 8), ResBlock(8, 8),
    nn.Upsample(scale_factor=2),
    MemoryLayer('#1', output=True), nn.Conv2d(16, 8, 1),
    ResBlock(8, 8), ResBlock(8, 8), ResBlock(8, 8), ResBlock(8, 8),
    MemoryLayer('#0', output=True), nn.Conv2d(14, 8, 1),
    ResBlock(8, 8), ResBlock(8, 8), ResBlock(8, 8), ResBlock(8, 8),
    nn.Conv2d(8, 6, 1),
)


# Optimizer helper (needed before creating opt)
def optimizer(parameters):
    return Adam(parameters, lr=2e-4)


In [10]:
# Cell 8: VAEAC class and create model instance
class VAEAC(nn.Module):
    def __init__(self, rec_log_prob, proposal_network, prior_network, generative_network, sigma_mu=1e4, sigma_sigma=1e-4):
        super().__init__()
        self.rec_log_prob = rec_log_prob
        self.proposal_network = proposal_network
        self.prior_network = prior_network
        self.generative_network = generative_network
        self.sigma_mu = sigma_mu
        self.sigma_sigma = sigma_sigma

    def make_observed(self, batch, mask):
        observed = batch.clone()
        observed[mask.bool()] = 0
        return observed

    def make_latent_distributions(self, batch, mask, no_proposal=False):
        observed = self.make_observed(batch, mask)
        if no_proposal:
            proposal = None
        else:
            full_info = torch.cat([batch, mask], 1)
            proposal_params = self.proposal_network(full_info)
            mu_p, sigma_p = parse_params_to_mu_sigma(proposal_params)
            proposal = (mu_p, sigma_p)
        prior_params = self.prior_network(torch.cat([observed, mask], 1))
        mu_prior, sigma_prior = parse_params_to_mu_sigma(prior_params)
        prior = (mu_prior, sigma_prior)
        return proposal, prior

    def prior_regularization(self, prior):
        mu = prior[0].view(prior[0].shape[0], -1)
        sigma = prior[1].view(prior[1].shape[0], -1)
        mu_reg = -(mu ** 2).sum(-1) / (2 * self.sigma_mu ** 2)
        sigma_reg = (sigma.log() - sigma).sum(-1) * self.sigma_sigma
        return mu_reg + sigma_reg

    def batch_vlb(self, batch, mask):
        proposal, prior = self.make_latent_distributions(batch, mask)
        mu_q, sigma_q = proposal
        mu_p, sigma_p = prior
        prior_reg = self.prior_regularization(prior)
        z = sample_from_mu_sigma(mu_q.float(), sigma_q.float())
        z = torch.clamp(z, -10.0, 10.0)     # stability fix
        # feed z into generative_network (z is 4D)
        rec_params = self.generative_network(z)
        # compute reconstruction log-prob using gaussian log-likelihood (manual)
        # rec_params are (B, 2*Cimg, Himg, Wimg); parse to mu_img, sigma_img
        rec_params = rec_params.clamp(-20, 20)   # avoid INF/NaN in decoder outputs
        mu_img, sigma_img = parse_params_to_mu_sigma(rec_params)
        # groundtruth flattened similarly
        gt = batch
        # compute per-pixel gaussian log-prob (elementwise)
        # log_prob = -0.5 * ((gt - mu)/sigma)^2 - log(sigma) - 0.5*log(2*pi)
        tol = 1e-6
        term = -0.5 * ((gt - mu_img) / (sigma_img + tol))**2 - torch.log(sigma_img + tol) - 0.5 * math.log(2*math.pi)
        # apply mask (mask==1 is missing region — in original code they multiply by mask to compute likelihood on all pixels in some setups)
        # We'll sum over all elements but only where mask==1 (masked region to evaluate)
        recon_log_prob = (term * mask).view(term.size(0), -1).sum(-1)

        # -------- KL divergence --------
        # KL( q(z|x) || p(z) ) for diagonal Gaussians
        kl = kl_normal(mu_q, sigma_q, mu_p, sigma_p)
        kl = torch.clamp(kl, -1e6, 1e6)     # safety

        # -------- Final VLB --------
        vlb = recon_log_prob - kl + prior_reg

        # Extra safety: prevent NaN returns
        if torch.isnan(vlb).any():
            print("⚠️ WARNING: NaN detected in VLB — returning zero.")
            return torch.zeros_like(vlb)

        return vlb

# instantiate reconstruction loss wrapper + model
reconstruction_log_prob = None  # handled inside batch_vlb now
model = VAEAC(reconstruction_log_prob, proposal_network, prior_network, generative_network)
print("Model instance created.")


Model instance created.


In [11]:
# Cell 9: move modules to GPU and prepare optimizer AFTER model instantiation
proposal_network = proposal_network.to(DEVICE)
prior_network = prior_network.to(DEVICE)
generative_network = generative_network.to(DEVICE)
model = model.to(DEVICE)

# safe mask wrapper (uses CPU-based ImageMaskGenerator internally)
class SafeImageMaskGenerator:
    def __init__(self):
        self.gen = ImageMaskGenerator()
    def __call__(self, batch):
        batch_cpu = batch.detach().cpu()
        mask_cpu = self.gen(batch_cpu)
        return mask_cpu.to(batch.device)

mask_generator = SafeImageMaskGenerator()

# create optimizer AFTER moving model to device
opt = optimizer(model.parameters())
scaler = GradScaler()
print("Model, networks, optimizer ready on", DEVICE)


Model, networks, optimizer ready on cuda


In [12]:
# Cell 10: pick a fixed batch for latent tracking
fixed_loader = DataLoader(train_ds, batch_size=SAVE_FIXED_COUNT, shuffle=False, num_workers=0)
fixed_batch = None
for fb in fixed_loader:
    fixed_batch = fb.to(DEVICE)
    with torch.no_grad():
        fixed_mask = mask_generator(fixed_batch)
    break
if fixed_batch is None:
    raise RuntimeError("Couldn't obtain fixed batch.")
print("Fixed batch for visualization:", fixed_batch.shape)


Fixed batch for visualization: torch.Size([16, 3, 128, 128])


In [13]:
# Cell 11: quick debug run (2 epochs, 10 batches) to ensure forward/backward executes
vlb_scale = 128 * 128
DEBUG_EPOCHS = 1
DEBUG_BATCHES = 8

for epoch in range(DEBUG_EPOCHS):
    model.train()
    pbar = tqdm(DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0), desc=f"Debug Epoch {epoch+1}")
    for i, batch in enumerate(pbar):
        if i >= DEBUG_BATCHES:
            break
        batch = batch.to(DEVICE)
        with torch.no_grad():
            mask = mask_generator(batch)
        with autocast(device_type='cuda' if DEVICE.type=='cuda' else 'cpu'):
            vlb = model.batch_vlb(batch, mask)
            loss = -vlb.mean() / vlb_scale
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.step(opt)
        scaler.update()
        pbar.set_postfix({"loss": f"{loss.item():.4f}"})
print("Debug run completed.")


Debug Epoch 1:   0%|          | 8/1881 [00:01<06:02,  5.17it/s, loss=20.4138]

Debug run completed.


In [14]:
# Cell 12: full training with latent tracking (fit PCA on first epoch then project)
vlb_scale = 128 * 128
all_latents = []
pca_model = None  # we will fit on epoch 1 and reuse

for epoch in range(1, EPOCHS+1):
    model.train()
    epoch_loss = 0.0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS}")
    for batch in pbar:
        batch = batch.to(DEVICE)
        with torch.no_grad():
            mask = mask_generator(batch)
        with autocast(device_type='cuda' if DEVICE.type=='cuda' else 'cpu'):
            vlb = model.batch_vlb(batch, mask)
            loss = -vlb.mean() / vlb_scale
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.step(opt)
        scaler.update()
        pbar.set_postfix({"loss": f"{loss.item():.4f}"})
        epoch_loss += loss.item() * batch.size(0)

    avg_loss = epoch_loss / len(train_ds)
    print(f"Epoch {epoch} avg loss: {avg_loss:.6f}")

    # --- compute z for fixed batch
    model.eval()
    with torch.no_grad():
        proposal, prior = model.make_latent_distributions(fixed_batch, fixed_mask)
        mu_q, sigma_q = proposal
        z = sample_from_mu_sigma(mu_q, sigma_q)  # (B,C,H,W)
        z_flat = z.view(z.size(0), -1).cpu().numpy()
        np.save(os.path.join(LATENT_DIR, f"epoch_{epoch:03d}_z_raw.npy"), z_flat)

        # PCA: fit on epoch 1, then project on later epochs for consistent axes
        if pca_model is None:
            pca_model = PCA(n_components=2)
            z_pca = pca_model.fit_transform(z_flat)
        else:
            z_pca = pca_model.transform(z_flat)

        np.save(os.path.join(LATENT_DIR, f"epoch_{epoch:03d}_z_pca.npy"), z_pca)
        all_latents.append(z_pca)

        # per-epoch PCA plot
        plt.figure(figsize=(6,5))
        plt.scatter(z_pca[:,0], z_pca[:,1], alpha=0.7)
        plt.title(f"Latent PCA — Epoch {epoch}")
        plt.xlabel("PC1"); plt.ylabel("PC2"); plt.grid(True)
        plt.tight_layout()
        plt.savefig(os.path.join(LATENT_DIR, f"epoch_{epoch:03d}_latent.png"), dpi=150)
        plt.close()

        # save predicted images for fixed batch
        proposal2, prior2 = model.make_latent_distributions(fixed_batch, fixed_mask)
        z2 = sample_from_mu_sigma(proposal2[0], proposal2[1])
        rec_params = model.generative_network(z2)
        mu_img, sigma_img = parse_params_to_mu_sigma(rec_params)
        preds = mu_img  # use mean as prediction
        x1_b = fixed_batch * (1 - fixed_mask)
        x_hat = x1_b + preds * fixed_mask
        x_hat = x_hat.clamp(-1, 1)

        epoch_dir = os.path.join(RESULTS_DIR, f"epoch_{epoch:03d}")
        os.makedirs(epoch_dir, exist_ok=True)
        for i in range(min(SAVE_FIXED_COUNT, x_hat.size(0))):
            sample_dir = os.path.join(epoch_dir, f"sample_{i:03d}")
            os.makedirs(sample_dir, exist_ok=True)
            save_image_tensor(os.path.join(sample_dir, "original.png"), fixed_batch[i])
            # mask: save first channel
            m = fixed_mask[i,0].detach().cpu().numpy()*255
            Image.fromarray(m.astype(np.uint8)).save(os.path.join(sample_dir, "mask.png"))
            save_image_tensor(os.path.join(sample_dir, "observed.png"), x1_b[i])
            save_image_tensor(os.path.join(sample_dir, "prediction.png"), x_hat[i])

# after training: combined latent evolution plot
plt.figure(figsize=(8,6))
cmap = plt.get_cmap("viridis")
for i, z_pca in enumerate(all_latents):
    color = cmap(i / max(1, len(all_latents)-1))
    plt.scatter(z_pca[:,0], z_pca[:,1], color=color, alpha=0.4)
plt.title("Latent evolution (PCA)")
plt.xlabel("PC1"); plt.ylabel("PC2")
plt.grid(True)
plt.tight_layout()
plt.savefig(os.path.join(LATENT_DIR, "latent_evolution.png"), dpi=200)
plt.close()

print("Training finished. Results:", RESULTS_DIR, LATENT_DIR)


Epoch 1/50: 100%|██████████| 1881/1881 [04:00<00:00,  7.83it/s, loss=0.7575]


Epoch 1 avg loss: 1.074704


Epoch 2/50: 100%|██████████| 1881/1881 [03:52<00:00,  8.08it/s, loss=0.5453]


Epoch 2 avg loss: 0.833730


Epoch 3/50: 100%|██████████| 1881/1881 [03:47<00:00,  8.28it/s, loss=0.5897]


...

Epoch 48 avg loss: -0.380855


Epoch 49/50: 100%|██████████| 1881/1881 [03:50<00:00,  8.15it/s, loss=-0.4344]


Epoch 49 avg loss: -0.383466


Epoch 50/50: 100%|██████████| 1881/1881 [03:50<00:00,  8.14it/s, loss=-0.3321]


Epoch 50 avg loss: -0.384074
Training finished. Results: C:\Preet\CV Project\VAE_AC\100 epochs\results C:\Preet\CV Project\VAE_AC\100 epochs\latent_history
